In [1]:
import numpy as np
from Heston import Heston

def check_variance_transition(model, dt=0.25, n_paths=100_000, seed=42):
    """Compare simulated variance moments with exact CIR moments."""
    rng = np.random.default_rng(seed)

    states = np.empty((n_paths, 2))
    states[:, 0] = model.s0
    states[:, 1] = model.v0

    next_states = model.step(states, dt, rng)
    variances = next_states[:, 1]

    decay = np.exp(-model.kappa * dt)
    one_minus_decay = -np.expm1(-model.kappa * dt)
    xi = model.vol_of_vol

    exact_mean = (
        model.theta + (model.v0 - model.theta) * decay
    )
    exact_variance = (
        model.v0 * xi**2 / model.kappa
        * decay * one_minus_decay
        + model.theta * xi**2 / (2.0 * model.kappa)
        * one_minus_decay**2
    )

    sample_mean = variances.mean()
    sample_variance = variances.var(ddof=1)

    assert np.all(np.isfinite(next_states))
    assert np.all(variances >= 0.0)
    np.testing.assert_array_equal(states[:, 0], model.s0)
    np.testing.assert_array_equal(states[:, 1], model.v0)

    print(f"Mean:     simulated={sample_mean:.8f}, exact={exact_mean:.8f}")
    print(
        f"Variance: simulated={sample_variance:.8f}, "
        f"exact={exact_variance:.8f}"
    )

    if exact_variance > 0.0:
        standard_error = np.sqrt(exact_variance / n_paths)
        z_score = (sample_mean - exact_mean) / standard_error

        print(f"Mean error in standard errors: {z_score:.2f}")
        print(
            "Relative variance error: "
            f"{(sample_variance / exact_variance - 1):.2%}"
        )
    else:
        np.testing.assert_allclose(
            variances,
            exact_mean,
            rtol=1e-12,
            atol=1e-14,
        )
        print("Deterministic variance check passed.")


parameters = dict(
    s0=100.0,
    v0=0.09,
    mu=0.0,
    kappa=2.0,
    theta=0.04,
    rho=-0.7,
)

for xi in (0.3, 0.8, 0.0):
    print(f"\nvol_of_vol = {xi}")
    model = Heston(**parameters, vol_of_vol=xi)
    check_variance_transition(model)


vol_of_vol = 0.3
Mean:     simulated=0.07044521, exact=0.07032653
Variance: simulated=0.00110941, exact=0.00110587
Mean error in standard errors: 1.13
Relative variance error: 0.32%

vol_of_vol = 0.8
Mean:     simulated=0.07034641, exact=0.07032653
Variance: simulated=0.00786661, exact=0.00786399
Mean error in standard errors: 0.07
Relative variance error: 0.03%

vol_of_vol = 0.0
Mean:     simulated=0.07032653, exact=0.07032653
Variance: simulated=0.00000000, exact=0.00000000
Deterministic variance check passed.


In [3]:
import numpy as np
import pandas as pd




def check_stock_mean(
    model,
    horizon=1.0,
    step_counts=(4, 12, 52, 252),
    n_paths=100_000,
    seed=42,
):
    """Check terminal stock means across simulation resolutions.

    Standard errors measure Monte Carlo uncertainty, not discretization bias.
    Only the current states are stored to limit memory use.
    """
    exact_mean = model.s0 * np.exp(model.mu * horizon)
    child_seeds = np.random.SeedSequence(seed).spawn(len(step_counts))
    results = []

    for n_steps, child_seed in zip(step_counts, child_seeds):
        rng = np.random.default_rng(child_seed)
        dt = horizon / n_steps

        states = np.empty((n_paths, 2))
        states[:, 0] = model.s0
        states[:, 1] = model.v0

        for _ in range(n_steps):
            states = model.step(states, dt, rng)

        terminal_prices = states[:, 0]

        if not np.all(np.isfinite(terminal_prices)):
            raise ValueError(f"Nonfinite stock prices with {n_steps} steps.")

        sample_mean = terminal_prices.mean()
        standard_error = terminal_prices.std(ddof=1) / np.sqrt(n_paths)
        error = sample_mean - exact_mean

        results.append({
            "n_steps": n_steps,
            "dt": dt,
            "sample_mean": sample_mean,
            "exact_mean": exact_mean,
            "error": error,
            "standard_error": standard_error,
            "error_in_SE": (
                error / standard_error if standard_error > 0 else np.nan
            ),
            "CI95_lower": sample_mean - 1.96 * standard_error,
            "CI95_upper": sample_mean + 1.96 * standard_error,
        })

    return pd.DataFrame(results)

In [4]:
model = Heston(
    s0=100.0,
    v0=0.09,
    mu=0.0,
    kappa=2.0,
    theta=0.04,
    vol_of_vol=0.8,
    rho=-0.7,
)

results = check_stock_mean(model)
display(results.round(6))

,n_steps,dt,sample_mean,exact_mean,error,standard_error,error_in_SE,CI95_lower,CI95_upper
0,4,0.250000,99.884408,100.0,-0.115592,0.068729,-1.681861,99.749700,100.019116
1,12,0.083333,99.960298,100.0,-0.039702,0.069042,-0.575045,99.824975,100.095620
2,52,0.019231,100.118660,100.0,0.118660,0.068780,1.725207,99.983851,100.253469
3,252,0.003968,100.121874,100.0,0.121874,0.068789,1.771703,99.987047,100.256700


In [8]:
import numpy as np
import pandas as pd
import QuantLib as ql


def heston_call_reference(model, strike, maturity_days=365):
    """Compute a Heston call benchmark at two integration resolutions."""
    if model.mu != 0.0:
        raise ValueError("This test requires mu=0.")
    if model.vol_of_vol <= 0 or model.theta <= 0:
        raise ValueError("This benchmark requires positive vol_of_vol and theta.")

    previous_date = ql.Settings.instance().evaluationDate

    try:
        today = ql.Date(1, 1, 2025)
        ql.Settings.instance().evaluationDate = today
        maturity = today + maturity_days
        day_count = ql.Actual365Fixed()
        horizon = day_count.yearFraction(today, maturity)

        zero_curve = ql.YieldTermStructureHandle(
            ql.FlatForward(today, 0.0, day_count)
        )

        process = ql.HestonProcess(
            zero_curve,  # Interest rate
            zero_curve,  # Dividend yield
            ql.QuoteHandle(ql.SimpleQuote(model.s0)),
            model.v0,
            model.kappa,
            model.theta,
            model.vol_of_vol,
            model.rho,
        )
        pricing_model = ql.HestonModel(process)

        option = ql.VanillaOption(
            ql.PlainVanillaPayoff(ql.Option.Call, strike),
            ql.EuropeanExercise(maturity),
        )

        prices = []
        for order in (96, 192):
            option.setPricingEngine(
                ql.AnalyticHestonEngine(pricing_model, order)
            )
            prices.append(option.NPV())

        return prices[-1], abs(prices[-1] - prices[0]), horizon

    finally:
        ql.Settings.instance().evaluationDate = previous_date


def check_call_prices(
    model,
    strikes=(80.0, 100.0, 120.0),
    maturity_days=365,
    step_counts=(4, 12, 52, 252),
    n_paths=100_000,
    seed=42,
):
    """Compare simulated call prices with analytical Heston benchmarks."""
    references = {
        strike: heston_call_reference(model, strike, maturity_days)
        for strike in strikes
    }
    horizon = references[strikes[0]][2]

    child_seeds = np.random.SeedSequence(seed).spawn(len(step_counts))
    rows = []

    for n_steps, child_seed in zip(step_counts, child_seeds):
        rng = np.random.default_rng(child_seed)
        dt = horizon / n_steps

        states = np.empty((n_paths, 2))
        states[:, 0] = model.s0
        states[:, 1] = model.v0

        for _ in range(n_steps):
            states = model.step(states, dt, rng)

        terminal_prices = states[:, 0]
        if not np.all(np.isfinite(terminal_prices)):
            raise ValueError("Simulation produced nonfinite stock prices.")

        for strike in strikes:
            reference, integration_change, _ = references[strike]
            payoffs = np.maximum(terminal_prices - strike, 0.0)

            estimate = payoffs.mean()
            se = payoffs.std(ddof=1) / np.sqrt(n_paths)
            error = estimate - reference

            rows.append({
                "n_steps": n_steps,
                "strike": strike,
                "MC_price": estimate,
                "reference_price": reference,
                "error": error,
                "standard_error": se,
                "error_in_SE": error / se if se > 0 else np.nan,
                "CI95_lower": estimate - 1.96 * se,
                "CI95_upper": estimate + 1.96 * se,
                "reference_integration_change": integration_change,
            })

    return pd.DataFrame(rows)

In [9]:
model = Heston(
    s0=100.0,
    v0=0.09,
    mu=0.0,
    kappa=2.0,
    theta=0.04,
    vol_of_vol=0.8,
    rho=-0.7,
)

results = check_call_prices(model)
display(results.round(6))

,n_steps,strike,MC_price,reference_price,error,standard_error,error_in_SE,CI95_lower,CI95_upper,reference_integration_change
0,4,80.0,22.706098,22.847009,-0.140912,0.051935,-2.713210,22.604304,22.807891,0.0
1,4,100.0,8.223868,8.301886,-0.078018,0.034155,-2.284213,8.156923,8.290812,0.0
2,4,120.0,1.291976,1.254344,0.037632,0.014695,2.560860,1.263174,1.320779,0.0
3,12,80.0,22.841748,22.847009,-0.005261,0.051667,-0.101825,22.740481,22.943016,0.0
4,12,100.0,8.294164,8.301886,-0.007722,0.033828,-0.228266,8.227860,8.360467,0.0
5,12,120.0,1.247259,1.254344,-0.007085,0.014467,-0.489731,1.218904,1.275614,0.0
6,52,80.0,22.936905,22.847009,0.089895,0.051742,1.737361,22.835490,23.038320,0.0
7,52,100.0,8.357750,8.301886,0.055864,0.033951,1.645412,8.291205,8.424295,0.0
8,52,120.0,1.257994,1.254344,0.003650,0.014638,0.249374,1.229303,1.286686,0.0
9,252,80.0,22.928926,22.847009,0.081917,0.051868,1.579329,22.827265,23.030588,0.0
